# 03 Knowledge Graph: A-Share Sector Event Propagation

This notebook demonstrates the **SectorGraph** built from 31 SW Level-1 industries.
It shows:
1. Build and inspect the sector graph
2. Propagate key events (semiconductor policy, new energy policy, real estate easing)
3. Visualize the full sector graph as an interactive force-directed network (Plotly)
4. Top-10 affected sectors table for '半导体扩产' scenario
5. Bar chart: propagation scores for selected event

In [3]:
import sys
import os
from pathlib import Path
repo_root = Path(os.getcwd()).resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / 'python'):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
print(f'Repo root: {repo_root}')


Repo root: /data00/home/guohuanwei.cztj/git_files/trade


In [4]:
import json
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from trade_py.analysis.knowledge_graph import (
    SectorGraph, SW, SW_NAMES_ZH,
    _EDGES, EVENT_SECTOR_MAPPING,
)
print('knowledge_graph module loaded')


knowledge_graph module loaded


## 1. Build and Inspect the Sector Graph

In [5]:
graph = SectorGraph()
d = graph.to_dict()

print(f"Nodes (sectors): {len(d['nodes'])}")
print(f"Edges:           {len(d['edges'])}")
print(f"Event types:     {len(d['event_mappings'])}")
print()
print('Available event types:')
for evt in sorted(graph.available_events()):
    print(f'  {evt}')

Nodes (sectors): 31
Edges:           61
Event types:     12

Available event types:
  commodity_slump
  commodity_surge
  defense_spending_up
  geopolitical_risk
  macro_recovery
  macro_slowdown
  new_energy_policy
  rate_cut
  rate_hike
  real_estate_easing
  real_estate_tightening
  semiconductor_policy


In [6]:
# Save graph JSON
graph_out = repo_root / 'data' / 'knowledge_graph' / 'sector_graph.json'
graph.save(graph_out)
print(f'Graph saved to: {graph_out}')

Graph saved to: /data00/home/guohuanwei.cztj/git_files/trade/data/knowledge_graph/sector_graph.json


In [7]:
# Edge breakdown by relation type
edges_df = pd.DataFrame(d['edges'])
rel_counts = edges_df['relation'].value_counts().reset_index()
rel_counts.columns = ['relation_type', 'count']
print(rel_counts.to_string(index=False))

    relation_type  count
  upstream_supply     25
   policy_linkage     12
    fund_rotation     10
   macro_exposure      6
downstream_demand      5
      competition      3


## 2. Event Propagation Examples

In [9]:
def show_propagation(event_type: str, max_hop: int = 2):
    results = graph.propagate_event(event_type, max_hop=max_hop)
    rows = []
    for r in results:
        rows.append({
            'sector_zh': r.sector_name,
            'score': round(r.score, 3),
            'hop': r.hop,
            'typical_days': r.typical_days,
            'relation': r.relation,
            'path': ' -> '.join(p.replace('SW_', '').replace(f'event:{event_type}', 'EVT')
                                for p in r.path),
        })
    df = pd.DataFrame(rows)
    pos = (df['score'] > 0).sum()
    neg = (df['score'] < 0).sum()
    print(f"\nEvent: {event_type}  |  {len(df)} sectors affected  "
          f"(+{pos} positive / -{neg} negative)")
    return df

In [10]:
# semiconductor_policy propagation
df_semi = show_propagation('semiconductor_policy')
df_semi


Event: semiconductor_policy  |  13 sectors affected  (+11 positive / -2 negative)


,sector_zh,score,hop,typical_days,relation,path
0,电子,0.900,1,0,primary,EVT -> Electronics
1,计算机,0.700,1,0,primary,EVT -> Computer
2,有色金属,0.600,1,0,primary,EVT -> NonFerrousMetal
3,国防军工,0.550,1,0,primary,EVT -> Defense
4,电气设备,0.500,1,0,primary,EVT -> ElectricalEquipment
5,通信,0.450,2,12,upstream_supply,EVT -> Electronics -> Telecom
6,汽车,0.405,2,12,upstream_supply,EVT -> Electronics -> Auto
7,机械设备,0.400,1,0,primary,EVT -> MechanicalEquipment
8,采掘,0.300,2,2,fund_rotation,EVT -> NonFerrousMetal -> Mining
9,传媒,0.280,2,12,policy_linkage,EVT -> Computer -> Media


In [11]:
# new_energy_policy propagation
df_energy = show_propagation('new_energy_policy')
df_energy


Event: new_energy_policy  |  16 sectors affected  (+11 positive / -5 negative)


,sector_zh,score,hop,typical_days,relation,path
0,电气设备,0.900,1,0,primary,EVT -> ElectricalEquipment
1,汽车,0.750,1,0,primary,EVT -> Auto
2,有色金属,0.700,1,0,primary,EVT -> NonFerrousMetal
3,电子,0.525,2,8,upstream_supply,EVT -> NonFerrousMetal -> Electronics
4,环保,0.500,1,0,primary,EVT -> Environment
5,煤炭,-0.500,1,0,primary,EVT -> Coal
6,化工,0.500,1,0,primary,EVT -> Chemical
7,国防军工,0.385,2,15,upstream_supply,EVT -> NonFerrousMetal -> Defense
8,公用事业,-0.375,2,5,upstream_supply,EVT -> Coal -> Utilities
9,石油石化,-0.350,1,0,primary,EVT -> Petroleum


In [12]:
# real_estate_easing propagation
df_realestate = show_propagation('real_estate_easing')
df_realestate


Event: real_estate_easing  |  15 sectors affected  (+15 positive / -0 negative)


,sector_zh,score,hop,typical_days,relation,path
0,房地产,0.900,1,0,primary,EVT -> RealEstate
1,建筑装饰,0.800,1,0,primary,EVT -> Construction
2,建筑材料,0.750,1,0,primary,EVT -> BuildingMaterial
3,家用电器,0.630,2,10,downstream_demand,EVT -> RealEstate -> HouseholdAppliance
4,银行,0.600,1,0,primary,EVT -> Banking
5,钢铁,0.500,1,0,primary,EVT -> Steel
6,非银金融,0.420,2,3,policy_linkage,EVT -> Banking -> NonBankFinancial
7,商业贸易,0.400,1,0,primary,EVT -> Commerce
8,机械设备,0.325,2,10,upstream_supply,EVT -> Steel -> MechanicalEquipment
9,轻工制造,0.315,2,15,downstream_demand,EVT -> RealEstate -> LightManufacturing


## 3. Interactive Sector Network Graph (Force-Directed, Plotly)

In [13]:
# Build NetworkX graph
G = nx.DiGraph()

for node in d['nodes']:
    G.add_node(node['id'], label=node['name_zh'], sw_code=node['sw_code'])

for edge in d['edges']:
    G.add_edge(
        edge['source'],
        edge['target'],
        relation=edge['relation'],
        weight=edge['weight'],
        direction=edge['direction'],
    )

print(f'NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Spring layout
import random
random.seed(42)
pos = nx.spring_layout(G, k=2.5, iterations=100, seed=42)

NetworkX graph: 31 nodes, 56 edges


In [14]:
# Color edges by relation type
RELATION_COLORS = {
    'upstream_supply':   '#1f77b4',  # blue
    'downstream_demand': '#ff7f0e',  # orange
    'policy_linkage':    '#2ca02c',  # green
    'fund_rotation':     '#9467bd',  # purple
    'macro_exposure':    '#8c564b',  # brown
    'competition':       '#d62728',  # red
}

fig = go.Figure()

# Draw edges grouped by relation type
edges_by_rel = {}
for src, tgt, data in G.edges(data=True):
    rel = data.get('relation', 'other')
    edges_by_rel.setdefault(rel, []).append((src, tgt, data))

for rel, edge_list in edges_by_rel.items():
    xe, ye = [], []
    for src, tgt, data in edge_list:
        x0, y0 = pos[src]
        x1, y1 = pos[tgt]
        xe += [x0, x1, None]
        ye += [y0, y1, None]
    color = RELATION_COLORS.get(rel, '#aaa')
    fig.add_trace(go.Scatter(
        x=xe, y=ye,
        mode='lines',
        line=dict(color=color, width=1.2),
        opacity=0.55,
        name=rel,
        legendgroup=rel,
        hoverinfo='none',
    ))

# Draw nodes
node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_labels = [G.nodes[n]['label'] for n in G.nodes()]
node_degrees = [G.degree(n) for n in G.nodes()]

fig.add_trace(go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    marker=dict(
        size=[8 + d * 1.5 for d in node_degrees],
        color=node_degrees,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Degree', thickness=12),
        line=dict(width=1, color='white'),
    ),
    text=node_labels,
    textposition='top center',
    textfont=dict(size=9),
    hovertext=[f"{G.nodes[n]['label']} ({n})" for n in G.nodes()],
    hoverinfo='text',
    name='Sector',
    showlegend=False,
))

fig.update_layout(
    title='A-Share Sector Knowledge Graph (SW Level-1)',
    showlegend=True,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=700,
    margin=dict(l=20, r=20, t=60, b=20),
    paper_bgcolor='#f8f9fa',
    plot_bgcolor='#f8f9fa',
)
fig.show()

## 4. Top-10 Affected Sectors: 半导体扩产 (Semiconductor Expansion)

In [15]:
# 'semiconductor_policy' models the semiconductor expansion scenario
results_semi = graph.propagate_event('semiconductor_policy', max_hop=2)

top10 = results_semi[:10]
rows = []
for i, r in enumerate(top10, 1):
    path_str = ' -> '.join(
        p.replace('SW_', '').replace('event:semiconductor_policy', '半导体政策')
        for p in r.path
    )
    rows.append({
        '排名': i,
        '板块': r.sector_name,
        '传导分数': f'{r.score:+.3f}',
        '传导跳数': r.hop,
        '预期滞后(日)': r.typical_days,
        '关系类型': r.relation,
        '传导路径': path_str,
    })

top10_df = pd.DataFrame(rows)
print('半导体扩产政策 -> 传导影响 Top 10 板块')
print(top10_df.to_string(index=False))

半导体扩产政策 -> 传导影响 Top 10 板块
 排名   板块   传导分数  传导跳数  预期滞后(日)            关系类型                               传导路径
  1   电子 +0.900     1        0         primary               半导体政策 -> Electronics
  2  计算机 +0.700     1        0         primary                  半导体政策 -> Computer
  3 有色金属 +0.600     1        0         primary           半导体政策 -> NonFerrousMetal
  4 国防军工 +0.550     1        0         primary                   半导体政策 -> Defense
  5 电气设备 +0.500     1        0         primary       半导体政策 -> ElectricalEquipment
  6   通信 +0.450     2       12 upstream_supply    半导体政策 -> Electronics -> Telecom
  7   汽车 +0.405     2       12 upstream_supply       半导体政策 -> Electronics -> Auto
  8 机械设备 +0.400     1        0         primary       半导体政策 -> MechanicalEquipment
  9   采掘 +0.300     2        2   fund_rotation 半导体政策 -> NonFerrousMetal -> Mining
 10   传媒 +0.280     2       12  policy_linkage         半导体政策 -> Computer -> Media


## 5. Bar Chart: Propagation Scores by Sector

In [16]:
def plot_propagation_scores(event_type: str, max_hop: int = 2, top_n: int = 20):
    results = graph.propagate_event(event_type, max_hop=max_hop)
    # Show top_n by abs score
    shown = results[:top_n]
    
    df = pd.DataFrame({
        'sector': [r.sector_name for r in shown],
        'score': [r.score for r in shown],
        'hop': [r.hop for r in shown],
    })
    df = df.sort_values('score')
    
    colors = ['#e84545' if s > 0 else '#2ba02b' for s in df['score']]
    
    fig = go.Figure(go.Bar(
        x=df['score'],
        y=df['sector'],
        orientation='h',
        marker_color=colors,
        text=[f"{s:+.2f}" for s in df['score']],
        textposition='outside',
        customdata=df['hop'],
        hovertemplate='%{y}<br>score: %{x:+.3f}<br>hop: %{customdata}<extra></extra>',
    ))
    fig.update_layout(
        title=f'Event: {event_type} — Sector Propagation Scores (top {top_n})',
        xaxis_title='Propagation Score',
        yaxis_title='Sector',
        height=max(400, len(df) * 28),
        xaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor='black'),
        margin=dict(l=120, r=80),
        showlegend=False,
    )
    fig.show()


plot_propagation_scores('semiconductor_policy')

In [17]:
plot_propagation_scores('new_energy_policy')

In [18]:
plot_propagation_scores('real_estate_easing')

## 6. Graph Reload Round-Trip Test

In [19]:
# Load from saved JSON and verify results match
graph2 = SectorGraph.load(graph_out)
r1 = graph.propagate_event('semiconductor_policy')
r2 = graph2.propagate_event('semiconductor_policy')

assert len(r1) == len(r2), f'Length mismatch: {len(r1)} vs {len(r2)}'
for a, b in zip(r1, r2):
    assert abs(a.score - b.score) < 1e-4, f'Score mismatch for {a.sector_name}: {a.score} vs {b.score}'

print('Round-trip load/save: OK')
print(f'  {len(r1)} sectors propagated from semiconductor_policy')

Round-trip load/save: OK
  13 sectors propagated from semiconductor_policy
